# Descarga de Datos

Como primera etapa del analisis exploratorio de datos. Vamos a descargar la información de la Direccion General del Servicio de Información Agroalimentaria y Pesquera. Para ello realizaremos el acceso mediante 'dgsiap.py'

In [6]:
from scripts import dgsiap
import re

Una vez establecida la conexión. Seguiremos con la selección de datos.

## Introducción

Primero crearemos un ligero mensaje para introducir la fuente de información.

In [7]:
# Definimos nuestra zona de trabajo.
content = dgsiap.page.find("main")
# Dividiremos la pagina en sus secciones principales
sections = content.find_all("section", recursive=False)

En este caso la pagina web nos permitio asociar cada bloque "section" con una bloque importante de la pagina del DGSIAP

* Datos generales de la fuente.
* Introducción breve
* Ultima publicación. (El conjunto de datos más reciente)
* Información por nivel. (Datos históricos)

En esta sección trabajaremos las primeras 2 categorias. El resto se verán en la descarga de los datos.

### Datos Generales

Explorando la pagina web podemos obtener información relevante para el analisis.

In [8]:
tree = sections[0]

intro_data = []
for t in tree.find_all("li"):
    if t.text!="":
        intro_data.append(t.text)

print(f"De parte de la {intro_data[0]} te damos la bienvenida al conjunto de {intro_data[1]} de la {intro_data[2]}")

De parte de la Dirección General del Servicio de Información Agroalimentaria y Pesquera te damos la bienvenida al conjunto de Datos abiertos de la Información agrícola


Como podemos ver en esta parte. El conjunto de datos forma parte de los datos Abiertos de la DGSIAP. Una dependencia del gobierno de México.

### Breve introducción.

El DGSIAP ya nos da un pequeño adelante de lo veremos en los datos. 

In [9]:
intro = sections[1]
raw_intro=list()
for t in intro.find_all("div", recursive=False):
    if t.text!="":
        raw_block=(t.text).split("\t")
        for raw in raw_block:
            if raw.strip() != "":
                raw_intro.append(raw)

print(f"Conjuntos de {raw_intro[0]} sobre {raw_intro[1]}")
print(raw_intro[-1])

Conjuntos de Datos abiertos sobre Estadística de Producción Agrícola

Los usuarios de la información podrán disponer de los datos anualizados de estadística básica agrícola a nivel nacional, estatal, por Distrito de Desarrollo Rural y municipal, la cual será de gran utilidad para el análisis del subsector y la toma de decisiones públicas y privadas que contribuyan al desarrollo rural sustentable del país.



Lo anterior solo es la introducción hacia el conjunto de datos. Ahora profundizaremos en los datos mediante los ultimos bloques.

* Última publicación.
* Información por nivel.

## Datos
Dentro del conjunto de datos se encontraran tablas en función de un cierto parámetro. El DGSIAP decidio clasificarlos en función del tipo.
* Información Principal. (Municipal y Nacional en la página web).
* No seguimiento. (Registro de elementos agricolas no prioritarios).

En el caso de la información principal está se divide por región
* Nivel Municipal.
* Nivel Nacional.
mientras la sección de no seguimiento solo posee a nivel nacional. De forma indistinta, cada una posee una subdivisión anual.

Cada bloque antes mencionado
* Nivel Municional.
* Nivel Nacional.
* No seguimiento 
posee un diccionario de datos que nos ayudara a interpretar mejor la información más adelante.

### Ultima publicación.
Como su nombre indica, corresponde a los datos más recientes. Para cada tabla de datos a descargar y usar debemos obtener su respectivo diccionario de datos.

In [38]:
last = sections[2]
search_title_section=last.find("div")
last_title = search_title_section.text.strip("\n").strip("\t")[:-1]
last_title

'Última publicación'

#### Diccionario de Datos.

Para conocer mejor la base de datos a trabajar comenzaremos obteniendo el diccionario de datos de la ultima publicación.

In [45]:
def get_catalog(sec=sections[2]):
    block = sec.find("div", {"class":"dict-section"})
    mini_blocks = block.find_all("div", {"class":re.compile(r"col-12 col-md-6*?")})
    codebook, metadata = mini_blocks[0], mini_blocks[1]
    codebook_info = {"title": codebook.find("h2").text, 
                     "description": codebook.find("p").text,
                     "download": codebook.find("a")['href']}
    meta = metadata.find("div")
    datetype = meta.find_all("div", recursive=False)
    metadata_info = {"title": meta.find("h4").text,
                     "state": datetype[0].find("span", {"class":"label"}).text,
                     "lastdate": datetype[0].find("span", {"class":"value"}).text,
                     "range": datetype[1].find("span", {"class":"label"}).text,
                     "size": datetype[1].find("span", {"class":"value"}).text
                    }
    
    return codebook_info, metadata_info

last_codebook, last_metadata = get_catalog()
print(f"{last_title}")
print(f"{last_codebook['title']}")
print(f"{last_metadata['state']} el dia {last_metadata['lastdate']}")
print(f"{last_codebook['description']}")

Última publicación
Diccionario de datos
Actualizado el dia 12 agosto 2026
Consulte cada nombre de campo, rangos válidos y la descripción de los campos que se utilizan en nuestras bases de datos.


#### Tablas de Datos.

La descarga de los diccionarios lo revisaremos durante esta sección para conglomerar todas las descargas. Asi que procedemos con la descarga de los archivos.